# Zebrafish: GitHub + scripts (API + SRA Toolkit)

We keep the **same repo** locally and on `sequoia:/home/zebrafish` so everyone runs the same scripts.
We use **two download approaches**: API for metadata/coordination, and SRA Toolkit for FASTQ generation.


## Why we keep both approaches

API = fast exploration + reproducible run lists; SRA Toolkit = standard implementation for producing FASTQs (fastq-dump / fasterq-dump).


## 0) Confirm local vs server repo are in sync

This prints the git commit hash locally and on the server (`/home/zebrafish`). If SSH fails, run `ssh pzg8794@sequoia.rit.edu` in a terminal once to set up keys/hostkey.



In [2]:
!ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu "hostname; whoami; pwd"


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.
sequoia
pzg8794
/home/pzg8794


In [3]:
%%bash
set -euo pipefail

# Compare local vs server git HEAD
GIT_ROOT="$(git rev-parse --show-toplevel)"
cd "$GIT_ROOT"

echo "LOCAL HEAD:  $(git rev-parse HEAD)"
git status -sb || true

echo
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"
$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

echo "SERVER HEAD: $(git rev-parse HEAD)"
git status -sb || true
EOF


LOCAL HEAD:  842a774482e2b3f7e217ae27f6584a2c228921f3
## main...origin/main
 M zebrafish/.gitignore
 M zebrafish/README.md
 M zebrafish/zebrafish_github_setup_and_script_walkthrough.ipynb
 M zebrafish/zebrafish_sra_api_test_download.ipynb
?? zebrafish/scripts/compare_fastq_subsets.py
?? zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh
?? zebrafish/scripts/download_runfiles_ncbi_download_path.py
?? zebrafish/scripts/ensure_sratoolkit.py
?? zebrafish/scripts/split_runs_among_members.py



RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


SERVER HEAD: 842a774482e2b3f7e217ae27f6584a2c228921f3
## main...origin/main
 M zebrafish/metadata/PRJNA1277581/runs.test5.smallest_sizeMB.txt
?? zebrafish/notes/tool_search/
?? zebrafish/tools/


## 1) Pull the latest repo on the server (fixes missing scripts)

Run this once when things look out of date or you see a “missing file” error.


In [ ]:
%%bash
set -euo pipefail

# Update server repo
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# If git refuses due to 'dubious ownership', run once:
#   git config --global --add safe.directory /home/zebrafish

git fetch origin
# Use ff-only to avoid accidental merges on the shared server

git pull --ff-only

echo "SERVER HEAD: $(git rev-parse HEAD)"
EOF


## 2) What scripts exist (shared by all members)

These are the scripts everyone should use (local or on the server).


In [ ]:
%%bash
set -euo pipefail

# List scripts locally + on server
echo "LOCAL scripts:" 
ls -la Semester5/BIOL550/group_project/zebrafish/scripts | sed -n '1,120p'

echo
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"
$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

echo "SERVER scripts:" 
ls -la zebrafish/scripts | sed -n '1,120p'
EOF


## API approach: metadata + coordination

We use the SRA RunInfo API to get run metadata and generate stable SRR lists for the team.


### A1) Script: `get_zebrafish_data_sra.py`

Fetches RunInfo (`runinfo.csv`) and writes SRR lists (all + filtered) under `zebrafish/metadata/<ACC>/`.


In [ ]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/get_zebrafish_data_sra.py --help | sed -n '1,120p'
EOF


### A2) Script: `download_runfiles_ncbi_download_path.py`

Downloads the run file for each SRR using the `download_path` column in `runinfo.csv` (no SRA Toolkit needed).


In [ ]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/download_runfiles_ncbi_download_path.py --help | sed -n '1,160p'
EOF

# Example (server): download runfiles for a run list
# python3 zebrafish/scripts/download_runfiles_ncbi_download_path.py #   --acc PRJNA1277581 #   --runs-file zebrafish/metadata/PRJNA1277581/runs.filtered.txt #   --runinfo-csv zebrafish/metadata/PRJNA1277581/runinfo.csv #   --base-dir zebrafish/data/runfiles


## SRA Toolkit approach: FASTQs (implementation)

We install SRA Toolkit once, then use it to convert SRRs into paired FASTQ files for analysis.


### S1) Install toolkit: `ensure_sratoolkit.py` (or Step 4a in the other notebook)

This keeps the toolkit in `zebrafish/tools/sratoolkit/` (gitignored) so server + local usage matches.


In [ ]:
%%bash
set -euo pipefail

# Install / ensure SRA Toolkit on the server
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/ensure_sratoolkit.py --repo-root "$PWD" --print-bin
EOF


### S2) Script: `download_fastq_sratoolkit_from_runs.sh`

Uses `prefetch` + `fasterq-dump --split-files --threads N` to download FASTQs from a runs file.


In [ ]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh --help | sed -n '1,160p'
EOF


## 3) Split the dataset among team members

This creates three run-list files (piter/nikhi/samuel) so downloads don’t overlap.


In [ ]:
%%bash
set -euo pipefail

# Split runs (server) into per-member files
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

OUT_DIR="zebrafish/metadata/$ACC/splits"
mkdir -p "$OUT_DIR"

# Prefer filtered list; fall back to all-runs list.
IN="zebrafish/metadata/$ACC/runs.filtered.txt"
[ -f "$IN" ] || IN="zebrafish/metadata/$ACC/runs.all.txt"

python3 zebrafish/scripts/split_runs_among_members.py   --runs-file "$IN"   --members piter nikhi samuel   --out-dir "$OUT_DIR"   --prefix runs.member

ls -la "$OUT_DIR"
EOF


## Member download: piter

Set `DO_RUN=1` to actually start downloading FASTQs for piter’s assigned SRRs.


In [ ]:
%%bash
set -euo pipefail

# piter: download FASTQs for your split SRR list (server)
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
DO_RUN=0

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC DO_RUN=$DO_RUN bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# Ensure toolkit on PATH
export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

RUNS_FILE="zebrafish/metadata/$ACC/splits/runs.member.piter.txt"
OUT_DIR="zebrafish/data/fastq/full/$ACC/piter"
THREADS=4

echo "runs_file: $RUNS_FILE"
echo "out_dir:   $OUT_DIR"
echo "threads:   $THREADS"

if [ "$DO_RUN" != "1" ]; then
  echo "DO_RUN=0 (dry). Set DO_RUN=1 in the notebook cell to start the download."
  exit 0
fi

bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh   --runs-file "$RUNS_FILE"   --out-dir "$OUT_DIR"   --threads "$THREADS"
EOF


## Member download: nikhi

Set `DO_RUN=1` to actually start downloading FASTQs for nikhi’s assigned SRRs.


In [ ]:
%%bash
set -euo pipefail

# nikhi: download FASTQs for your split SRR list (server)
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
DO_RUN=0

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC DO_RUN=$DO_RUN bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# Ensure toolkit on PATH
export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

RUNS_FILE="zebrafish/metadata/$ACC/splits/runs.member.nikhi.txt"
OUT_DIR="zebrafish/data/fastq/full/$ACC/nikhi"
THREADS=4

echo "runs_file: $RUNS_FILE"
echo "out_dir:   $OUT_DIR"
echo "threads:   $THREADS"

if [ "$DO_RUN" != "1" ]; then
  echo "DO_RUN=0 (dry). Set DO_RUN=1 in the notebook cell to start the download."
  exit 0
fi

bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh   --runs-file "$RUNS_FILE"   --out-dir "$OUT_DIR"   --threads "$THREADS"
EOF


## Member download: samuel

Set `DO_RUN=1` to actually start downloading FASTQs for samuel’s assigned SRRs.


In [ ]:
%%bash
set -euo pipefail

# samuel: download FASTQs for your split SRR list (server)
SSH="ssh -o BatchMode=yes -o ConnectTimeout=10 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
DO_RUN=0

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC DO_RUN=$DO_RUN bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# Ensure toolkit on PATH
export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

RUNS_FILE="zebrafish/metadata/$ACC/splits/runs.member.samuel.txt"
OUT_DIR="zebrafish/data/fastq/full/$ACC/samuel"
THREADS=4

echo "runs_file: $RUNS_FILE"
echo "out_dir:   $OUT_DIR"
echo "threads:   $THREADS"

if [ "$DO_RUN" != "1" ]; then
  echo "DO_RUN=0 (dry). Set DO_RUN=1 in the notebook cell to start the download."
  exit 0
fi

bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh   --runs-file "$RUNS_FILE"   --out-dir "$OUT_DIR"   --threads "$THREADS"
EOF
